# Steam Game Recommendation: Playtime-Aware BPR

**CSE 258, Assignment 2**

**Research question:** Does playtime-weighted Bayesian Personalized Ranking (BPR) outperform standard BPR on top-K game recommendation?

The notebook follows the five graded sections:

1. Predictive task and evaluation protocol
2. Dataset, preprocessing and exploratory analysis
3. Modeling
4. Evaluation and results
5. Related work and limitations

<!-- TEAM NOTES (delete before export)
Owners: [S] = Svante, [T] = Tony, [S+T] = both. Only edit cells in your own sections.
Pull before editing, commit small changes often, clear outputs before committing.
See doc.md for the full plan and integration contract. -->

## 0. Setup

In [ ]:
import ast
import gzip
import json
import math
import os
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")
for d in (DATA_DIR, RESULTS_DIR, FIGURES_DIR):
    d.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Predictive task and evaluation protocol  <!-- [S] -->

**Task.** Given a user's observed Steam library and playtime, rank games outside that library. The target is held-out game *ownership*, not future purchases, ratings or hours played.

**Evaluation.** TODO: per-user 80/10/10 split, full-catalog ranking with training (and validation) games masked, macro-averaged Recall@10, Recall@20, NDCG@10 and AUC.

**Baselines.** TODO: popularity, item-based Jaccard, standard BPR.

**Validity.** TODO: validation-only model selection, untouched test set, identical candidate sets across models, fixed seeds.

### 1.1 Evaluation functions  <!-- [S] -->

Defined here so they can be used for BPR early stopping in Section 3. Every model exposes `score_users(user_ids) -> np.ndarray` of shape `(len(user_ids), num_items)`, with scores in shared item-ID order.

In [ ]:
def evaluate(model, eval_df, mask_dfs, num_items, ks=(10, 20), batch_size=1024):
    """Rank all train-supported items for each eligible user and compute metrics.

    model:    object with score_users(user_ids) -> (n_users, num_items) scores
    eval_df:  held-out interactions (val or test) with columns user_id, item_id
    mask_dfs: interactions to exclude from ranking (train for val; train + val for test)
    Returns a dict of macro-averaged Recall@k, NDCG@10 and AUC, plus per-user
    metrics for the activity-group analysis in Section 4.
    """
    raise NotImplementedError  # TODO [S]

## 2. Dataset, preprocessing and exploratory analysis  <!-- [S] -->

**Source.** Steam data, Version 1, user and item data (Pathak, Gupta and McAuley, 2017): https://cseweb.ucsd.edu/~jmcauley/datasets.html#steam_data

TODO: context on how the data was collected, and why ownership is imperfect implicit feedback from a snapshot rather than a purchase history.

### 2.1 Download and parse  <!-- [S] -->

In [ ]:
RAW_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/steam/australian_users_items.json.gz"
RAW_PATH = DATA_DIR / "australian_users_items.json.gz"

if not RAW_PATH.exists():
    print("Downloading (~74 MB)...")
    urllib.request.urlretrieve(RAW_URL, RAW_PATH)

# Each line is a Python dict literal (not strict JSON), so parse with ast.literal_eval.
# TODO [S]: flatten each user's `items` list into rows of (user_id, item_id, playtime_forever).

### 2.2 Cleaning and filtering  <!-- [S] -->

In [ ]:
# TODO [S]: remove duplicate user-game pairs, validate nonnegative playtime,
# keep zero-playtime rows, convert minutes to hours, then iteratively filter
# users with < 10 games and games with < 5 users. Report counts before and after.

### 2.3 Train / validation / test split  <!-- [S] -->

In [ ]:
# TODO [S]: seeded per-user 80/10/10 split (at least one interaction per partition),
# contiguous integer IDs, drop held-out items not seen in training (report how many).
# Save the split so model cells can reload it without rerunning preprocessing:
#   data/train.parquet, data/val.parquet, data/test.parquet
#       columns: user_id (int), item_id (int), playtime_hours (float)
#   data/meta.json: num_users, num_items, seed, ID mappings

### 2.4 Exploratory analysis  <!-- [S] -->

In [ ]:
# TODO [S]: user/item/interaction counts, sparsity, zero-playtime fraction,
# playtime quantiles; plots of games per user, users per game, raw and log playtime.

## 3. Modeling  <!-- [S+T] -->

TODO: formulation (inputs, outputs, objective) and the trade-offs between the models (complexity, efficiency, implementation challenges).

### 3.0 Load the shared split  <!-- [S] -->

In [ ]:
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
val_df = pd.read_parquet(DATA_DIR / "val.parquet")
test_df = pd.read_parquet(DATA_DIR / "test.parquet")
with open(DATA_DIR / "meta.json") as f:
    meta = json.load(f)
num_users, num_items = meta["num_users"], meta["num_items"]

### 3.1 Popularity baseline  <!-- [S] -->

Score each game by its number of distinct training users.

In [ ]:
class Popularity:
    def __init__(self, train_df, num_items):
        raise NotImplementedError  # TODO [S]

    def score_users(self, user_ids):
        raise NotImplementedError  # TODO [S]

### 3.2 Item-based Jaccard similarity  <!-- [S] -->

$J(i,j)=\frac{|U_i\cap U_j|}{|U_i\cup U_j|}$. A candidate's score is the sum of its similarities to the games in the user's training history.

In [ ]:
class ItemJaccard:
    def __init__(self, train_df, num_users, num_items, top_k_neighbors=100):
        raise NotImplementedError  # TODO [S]

    def score_users(self, user_ids):
        raise NotImplementedError  # TODO [S]

### 3.3 Standard BPR  <!-- [T] -->

$s(u,i)=p_u^\top q_i$, trained on triplets $(u,i,j)$ with $i$ a training-owned game and $j$ a sampled game outside the user's training history.

$\mathcal L=\mathbb E[-\log\sigma(s(u,i)-s(u,j))]+\lambda\|\Theta\|_2^2$

In [ ]:
class BPR(torch.nn.Module):
    def __init__(self, num_users, num_items, dim=64):
        super().__init__()
        raise NotImplementedError  # TODO [T]

    def score_users(self, user_ids):
        raise NotImplementedError  # TODO [T]: return a numpy array (n_users, num_items)


def train_bpr(train_df, val_df, num_users, num_items, weights=None, dim=64, lr=1e-3,
              reg=1e-4, batch_size=4096, epochs=30, seed=SEED):
    """Train BPR with optional per-interaction weights (None = standard BPR).
    Uses softplus(s_neg - s_pos) and early stopping on validation NDCG@10."""
    raise NotImplementedError  # TODO [T]

### 3.4 Playtime-weighted BPR  <!-- [T] -->

$\mathcal L=\mathbb E[-w_{ui}\log\sigma(s(u,i)-s(u,j))]+\lambda\|\Theta\|_2^2$, with $a_{ui}=(1+\log(1+t_{ui}))^\alpha$.

| Variant | Raw weight |
|---|---|
| Standard BPR | 1 |
| Log-weighted BPR | $a_{ui}$ |
| User-normalized BPR | $a_{ui}/\operatorname{mean}_{k\in I_u^{train}}a_{uk}$ |

Weights are normalized to mean 1 over training interactions.

In [ ]:
def playtime_weights(train_df, alpha=1.0, user_normalize=False):
    """Return one weight per training interaction, normalized to mean 1."""
    raise NotImplementedError  # TODO [T]

## 4. Evaluation and results  <!-- [S+T] -->

TODO: why these metrics suit the task, and how the results show whether playtime weighting helps.

### 4.1 Experiment A: main comparison (test set)  <!-- [S+T] -->

In [ ]:
# TODO: train/fit all five models, evaluate on test, show the results table,
# and save results/main_comparison.csv (model, seed, split, metrics, config).

### 4.2 Experiment B: playtime sensitivity (alpha sweep, validation)  <!-- [T] -->

In [ ]:
# TODO [T]: alpha in {0, 0.25, 0.5, 1, 2}; plot validation NDCG@10 vs alpha.

### 4.3 Experiment C: user activity groups  <!-- [S+T] -->

In [ ]:
# TODO: split users by total training playtime (bottom 25%, middle 50%, top 25%),
# report group sizes and per-group metrics for standard vs weighted BPR.

## 5. Related work and limitations  <!-- [S+T] -->

- Rendle et al. (2009), *BPR: Bayesian Personalized Ranking from Implicit Feedback*. TODO [T]
- Hu, Koren and Volinsky (2008), *Collaborative Filtering for Implicit Feedback Datasets*. TODO [T]
- Pathak, Gupta and McAuley (2017), *Generating and Personalizing Bundle Recommendations on Steam*. TODO [S]

TODO: how our results compare with these works, and the limitations of the study.